# 91250 - Deep Learning 

## 0. Import and Configs

In [1]:
# File Operations
import gdown
import os
from pathlib import Path

# Math & Visualization
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau

# Misc
from sklearn.model_selection import train_test_split
import copy
import pprint as pp
import time

In [2]:
# Set Device to GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [3]:
# Fixing seed to reduce randomness, for better comparisons
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [4]:
# File Paths
TRAIN_FILE_ID = "1xyggntfZ2-6BTAagxJm-tFKDXerTAs8c"
TEST_FILE_ID  = "1VozRpr3dlVpA17BL-7ALCaOXa2vCe64J"

DATA_FOLDER = "data"
TRAIN_PATH = os.path.join(DATA_FOLDER,"chess_error_detection_train.csv")
TEST_PATH  = os.path.join(DATA_FOLDER,"chess_error_detection_test.csv")

MODEL_FOLDER = "model"
os.makedirs(MODEL_FOLDER, exist_ok=True)
BEST_ENCODER_PATH = os.path.join(MODEL_FOLDER,"best_encoder.pt")
BEST_MODEL_PATH = os.path.join(MODEL_FOLDER,"best_model.pt")

if not os.path.exists(TRAIN_PATH):
    gdown.download(id=TRAIN_FILE_ID, output=TRAIN_PATH, quiet=False)

if not os.path.exists(TEST_PATH):
    gdown.download(id=TEST_FILE_ID, output=TEST_PATH, quiet=False)

In [5]:
RUN_PRETRAINING = True
RUN_TRAINING = True

## Data

In [6]:
train_df = pd.read_csv(TRAIN_PATH)
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["error_position"]
)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape: ", test_df.shape)

display(train_df.head())

Train shape: (320000, 3)
Validation shape: (80000, 3)
Test shape:  (50000, 3)


,sequence,error_position,correct_move
91975,g3 d5 e4 c4 Nf3 Bg4 Bg2 e5 O-O Bd6 d3 Nf6 c4 d...,4,d4
252136,d4 e6 c4 d5 cxd5 Qxd5 e3 c5 Nc3 Qh5 Be2 Qg6 dx...,2,Nf6
264691,e4 e6 Qf3 c5 exd5 exd5 Nc3 Nf6 h3 Bb4 g4 O-O g...,4,d5
130957,e4 e5 Nf3 d6 Nf3 exd4 Nxd4 d5 exd5 Qxd5 Nc3 Qe...,5,d4
240371,e4 g6 d4 Bg7 Nc3 dxe4 Nge2 c6 h3 Qc7 g4 e5 Bg2...,6,d6


## Tokenization

In [7]:
unique_moves = set()
unique_moves.add("<PAD>")
unique_moves.add("<UNK>")

for game in train_df["sequence"]:
    for move in game.split():
        unique_moves.add(move)

print(len(unique_moves))

3597


In [8]:
train_vocab = unique_moves

test_moves = []

for game in test_df["sequence"]:
    test_moves.extend(game.split())

unknown_unique = set(test_moves) - train_vocab
print("Unique unseen moves:", len(unknown_unique))

unknown_count = sum(move not in train_vocab for move in test_moves)
print("Unknown count:", unknown_count)
print("Unknown rate:", unknown_count / len(test_moves))

Unique unseen moves: 99
Unknown count: 105
Unknown rate: 5.25e-05


In [9]:
vocab_to_id = {
    "<PAD>": 0,
    "<UNK>": 1
}

for move in unique_moves:
    if move not in vocab_to_id:
        vocab_to_id[move] = len(vocab_to_id)

VOCAB_SIZE = len(vocab_to_id)
print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 3597


In [10]:
id_to_vocab = {
    idx: move
    for move, idx in vocab_to_id.items()
}

In [11]:
def encode_sequence(sequence, vocab_to_id):
    return [
        vocab_to_id.get(move, vocab_to_id["<UNK>"])
        for move in sequence.split()
    ]

In [12]:
sequence = train_df["sequence"].iloc[0]

encoded = encode_sequence(sequence, vocab_to_id)

print(sequence)
print()
print(encoded)
print()
print(f"Length: {len(encoded)}")

g3 d5 e4 c4 Nf3 Bg4 Bg2 e5 O-O Bd6 d3 Nf6 c4 dxc3 Nxc3 c6 Bg5 O-O Qd2 Nbd7 Bxf6 Qxf6 Nh4 g5 Nf5 Bb4 a3 Bxc3 Qxc3 Nb6 Ne3 Bf3 Bxf3 Qxf3 Ng2 Qf6 f4 gxf4 Nxf4 Na4

[2131, 2894, 487, 1170, 2731, 3058, 548, 3209, 3439, 801, 1483, 3570, 1170, 3590, 2978, 945, 2338, 3439, 3191, 3167, 109, 2035, 40, 1111, 2861, 490, 1838, 2527, 2849, 2178, 1347, 1190, 3306, 837, 2871, 2420, 3506, 3040, 776, 941]

Length: 40


In [13]:
train_encoded = [
    encode_sequence(seq, vocab_to_id)
    for seq in train_df["sequence"]
]

val_encoded = [
    encode_sequence(seq, vocab_to_id)
    for seq in val_df["sequence"]
]

test_encoded = [
    encode_sequence(seq, vocab_to_id)
    for seq in test_df["sequence"]
]

In [14]:
train_labels = train_df["error_position"].to_numpy() - 1
val_labels = val_df["error_position"].to_numpy() - 1
test_labels = test_df["error_position"].to_numpy() - 1

In [15]:
x_train = torch.tensor(train_encoded, dtype=torch.long)
y_train = torch.tensor(train_labels, dtype=torch.long)

x_val = torch.tensor(val_encoded, dtype=torch.long)
y_val = torch.tensor(val_labels, dtype=torch.long)

x_test = torch.tensor(test_encoded, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

In [16]:
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

## Model Definitions

### Encoder

In [17]:
class ChessEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_layers=3,
        dim_feedforward=256,
        dropout=0.1,
        max_len=40
    ):
        super().__init__()
        
        self.d_model = d_model
        self.max_len = max_len

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            max_len,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        batch_size, seq_len = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        )

        token_embeddings = self.token_embedding(x)
        position_embeddings = self.position_embedding(positions)

        x = token_embeddings + position_embeddings

        x = self.transformer(x)
        return x

### Reconstructor 

In [18]:
class ChessReconstructor(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer ,
            num_layers=num_layers
        )

        self.output_projection = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, x):
        x = self.transformer(x)
        logits = self.output_projection(x)
        return logits

### Autoencoder

In [19]:
class ChessAutoencoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        encoder_layers=3,
        reconstructor_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        max_len=40,
        candidate_len=10,
        mask_ratio=0.3,
        pad_token_id=0
    ):
        super().__init__()

        self.candidate_len = candidate_len
        self.mask_ratio = mask_ratio
        self.pad_token_id = pad_token_id

        self.encoder = ChessEncoder(
            vocab_size=vocab_size,
            d_model=d_model,
            nhead=nhead,
            num_layers=encoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            max_len=max_len
        )

        self.reconstructor = ChessReconstructor(
            vocab_size=vocab_size,
            d_model=d_model,
            nhead=nhead,
            num_layers=reconstructor_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )

    # Create mask, always mask corrupted position and additional positions to make up for the masking ratio, 
    # Only the first 10 positions can be masked 
    def create_mask(
        self,
        batch_size,
        corrupted_positions,
        device
    ):
        # Number of positions to mask
        num_masked = max(
            1,
            round(self.candidate_len * self.mask_ratio)
        )

        mask = torch.zeros(
            batch_size,
            self.candidate_len,
            dtype=torch.bool,
            device=device
        )

        batch_indices = torch.arange(
            batch_size,
            device=device
        )

        # Always mask the known corrupted position
        mask[
            batch_indices,
            corrupted_positions
        ] = True

        # Number of additional positions
        additional_needed = num_masked - 1
        if additional_needed > 0:
            for i in range(batch_size):
            
                # Positions not already masked
                available = torch.where(
                    ~mask[i]
                )[0]

                # Randomly choose additional positions
                selected = available[
                    torch.randperm(
                        len(available),
                        device=device
                    )[:additional_needed]
                ]

                mask[i, selected] = True

        return mask

    def forward(self, x, corrupted_positions):
        mask = self.create_mask(
            batch_size=x.size(0),
            corrupted_positions=corrupted_positions,
            device=x.device
        )

        # Expand the mask to the entire move sequence
        full_mask = torch.zeros(
            x.size(0),
            x.size(1),
            dtype=torch.bool,
            device=x.device
        )
        full_mask[:, :self.candidate_len] = mask

        masked_x = x.clone()
        masked_x[full_mask] = self.pad_token_id

        encoded = self.encoder(masked_x)
        logits = self.reconstructor(encoded)

        return logits, full_mask

### Classifier

In [20]:
class ChessClassifier(nn.Module):
    def __init__(
        self,
        encoder,
        d_model=128,
        candidate_len=10
    ):
        super().__init__()

        self.encoder = encoder
        self.candidate_len = candidate_len

        self.classifier = nn.Linear(
            d_model,
            1
        )

    def forward(self, x):

        # Encode the entire game
        encoded = self.encoder(x)

        # Only the first 10 positions are candidates
        encoded = encoded[:, :self.candidate_len, :]

        # Predict one score for each candidate position
        logits = self.classifier(encoded)

        # [batch, 10, 1] -> [batch, 10]
        logits = logits.squeeze(-1)

        return logits

### Pre-training

In [21]:
autoencoder = ChessAutoencoder(vocab_size=VOCAB_SIZE)

In [22]:
autoencoder = autoencoder.to(DEVICE)

In [23]:
def reconstruction_loss(
    logits,
    targets,
    mask,
    corrupted_positions
):
    
    batch_size = targets.size(0)

    # Create mask for corrupted positions
    corrupted_mask = torch.zeros_like(mask)

    batch_indices = torch.arange(
        batch_size,
        device=targets.device
    )

    corrupted_mask[
        batch_indices,
        corrupted_positions
    ] = True

    # Only calculate loss on masked positions that are NOT corrupted
    loss_mask = mask & ~corrupted_mask

    # Select predictions and targets
    selected_logits = logits[loss_mask]
    selected_targets = targets[loss_mask]

    loss = F.cross_entropy(
        selected_logits,
        selected_targets
    )

    return loss

In [24]:
if(RUN_PRETRAINING):
    optimizer = torch.optim.AdamW(
        autoencoder.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.1,
        patience=3,
        min_lr=1e-6
    )

In [25]:
def train_autoencoder(model, loader, optimizer, DEVICE):
    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for x, corrupted_positions in loader:

        x = x.to(DEVICE)
        corrupted_positions = corrupted_positions.to(DEVICE)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        logits, mask = model(
            x,
            corrupted_positions
        )

        # Reconstruction loss
        loss = reconstruction_loss(
            logits,
            x,
            mask,
            corrupted_positions
        )

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        # Stats
        batch_size = x.size(0)
        total_loss += loss.item() * batch_size

        # Identify positions used for loss
        corrupted_mask = torch.zeros_like(mask)

        batch_indices = torch.arange(
            batch_size,
            device=DEVICE
        )

        corrupted_mask[
            batch_indices,
            corrupted_positions
        ] = True

        loss_mask = mask & ~corrupted_mask

        # Predictions at the reconstruction positions
        predictions = logits.argmax(dim=-1)
        total_correct += (predictions[loss_mask] == x[loss_mask]).sum().item()
        total_samples += loss_mask.sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = (
        total_correct / total_samples
        if total_samples > 0
        else 0
    )

    return avg_loss, accuracy

In [26]:
def validate_autoencoder(
    model,
    loader,
    DEVICE
):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():

        for x, corrupted_positions in loader:

            x = x.to(DEVICE)
            corrupted_positions = corrupted_positions.to(DEVICE)

            logits, mask = model(
                x,
                corrupted_positions
            )

            loss = reconstruction_loss(
                logits,
                x,
                mask,
                corrupted_positions
            )

            batch_size = x.size(0)

            total_loss += loss.item() * batch_size

            # Create mask for corrupted positions
            corrupted_mask = torch.zeros_like(mask)

            batch_indices = torch.arange(
                batch_size,
                device=DEVICE
            )

            corrupted_mask[
                batch_indices,
                corrupted_positions
            ] = True

            # Only evaluate the clean masked positions
            loss_mask = mask & ~corrupted_mask

            predictions = logits.argmax(dim=-1)

            total_correct += (
                predictions[loss_mask] == x[loss_mask]
            ).sum().item()

            total_samples += loss_mask.sum().item()

    avg_loss = total_loss / len(loader)

    accuracy = (
        total_correct / total_samples
        if total_samples > 0
        else 0
    )

    return avg_loss, accuracy

In [27]:
PATIENCE = 5
MAX_EPOCH = 200

# Train flags
best_val_loss = float('inf')
epochs_without_improvement = 0

# For plotting graph
train_loss_list = []
val_loss_list = []
epoch_times = []

In [ ]:
if(RUN_PRETRAINING):
    for epoch in range(MAX_EPOCH):
    
        start_time = time.time()
    
        train_loss, train_acc = train_autoencoder(
            autoencoder,
            train_loader,
            optimizer,
            DEVICE
        )
    
        val_loss, val_acc = validate_autoencoder(
            autoencoder,
            val_loader,
            DEVICE
        )
    
        scheduler.step(val_loss)
    
        # Store loss list
        train_loss_list.append(train_loss)
        val_loss_list.append(val_loss)
    
        # Calculate epoch time
        epoch_time = time.time() - start_time
    
        # Track LR
        current_lr = optimizer.param_groups[0]["lr"]
    
        # Print logging statements
        print(
            f"Epoch {epoch + 1}/{MAX_EPOCH} "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f} "
            f"| LR: {current_lr:.6f} "
            f"| Time: {epoch_time:.2f}s"
        )
    
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
    
            # Save best encoder
            torch.save(
                autoencoder.encoder.state_dict(),
                BEST_ENCODER_PATH
            )
    
        else:
            epochs_without_improvement += 1
    
            if epochs_without_improvement >= PATIENCE:
                print("Early stopping triggered.")
                break

In [ ]:
if(RUN_PRETRAINING):
    epochs = range(1, len(train_loss_list) + 1)
    
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss_list, label="Train Loss")
    plt.plot(epochs, val_loss_list, label="Validation Loss")
    
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Extract the encoder
if(RUN_PRETRAINING):
    pretrained_encoder = autoencoder.encoder

### Training

In [ ]:
if(RUN_PRETRAINING):
    classifier = ChessClassifier(
        encoder=pretrained_encoder,
        d_model=128,
        candidate_len=10
    ).to(DEVICE)
elif(RUN_TRAINING):
    encoder = ChessEncoder(
        vocab_size=VOCAB_SIZE,
        d_model=128,
        nhead=4,
        num_layers=3,
        dim_feedforward=256,
        dropout=0.1,
        max_len=40
    )
    
    encoder.load_state_dict(
        torch.load(BEST_ENCODER_PATH, map_location=DEVICE)
    )
    
    classifier = ChessClassifier(
        encoder=encoder,
        d_model=128,
        candidate_len=10
    )

In [ ]:
def train_classifier(
    model,
    loader,
    optimizer,
    criterion,
    DEVICE
):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # Reset gradient per batch
        optimizer.zero_grad()


        logits = model(x)
        loss = criterion(logits, y)

        # Backprop
        loss.backward()

        optimizer.step()

        # Stats
        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        predictions = logits.argmax(dim=1)

        total_correct += (predictions == y).sum().item()
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy

In [ ]:
def validate_classifier(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            predictions = logits.argmax(dim=1)

            total_correct += (
                predictions == y
            ).sum().item()

            total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy

In [ ]:
if(RUN_TRAINING):
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.AdamW(
        classifier.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.1,
        patience=3,
        min_lr=1e-6
    )

In [ ]:
PATIENCE = 5
MAX_EPOCH = 200

# Train flags
best_val_acc = 0.0
epochs_without_improvement = 0

# For plotting graphs
train_acc_list = []
val_acc_list = []
epoch_times = []

In [ ]:
if(RUN_TRAINING):
    for epoch in range(MAX_EPOCH):
        epoch_start_time = time.time()
    
        train_loss, train_acc = train_classifier(
            classifier,
            train_loader,
            optimizer,
            criterion,
            DEVICE
        )
    
        val_loss, val_acc = validate_classifier(
            classifier,
            val_loader,
            criterion,
            DEVICE
        )
    
        # Reduce LR when validation accuracy plateaus
        scheduler.step(val_acc)
        
        # Log accuracy
        train_acc_list.append(train_acc)
        val_acc_list.append(val_acc)
    
        # Calculate epoch time
        epoch_time = time.time() - epoch_start_time
        
        # Print logging statements
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch + 1}/{MAX_EPOCH} "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f} "
            f"| LR: {current_lr:.6f} "
            f"| Time: {epoch_time:.2f}s"
        )
    
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            
            torch.save(
                classifier.state_dict(),
                BEST_MODEL_PATH
            )
    
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print("Early stopping triggered.")
                break

In [ ]:
# Plot training and validation accuracy
if(RUN_TRAINING):
    epochs = range(1, len(train_acc_list) + 1)
    
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_acc_list, label="Train Accuracy")
    plt.plot(epochs, val_acc_list, label="Validation Accuracy")
    
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## Final Evaluation

In [ ]:
classifier = ChessClassifier(
    encoder=encoder,
    d_model=128,
    candidate_len=10
)

classifier.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=DEVICE)
)

classifier = classifier.to(DEVICE)

In [ ]:
# Given from project specification
def accuracy_at_1(y_true, probabilities):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    assert probabilities.ndim == 2
    assert probabilities.shape[1] == 10
    assert len(y_true) == len(probabilities)

    # argmax returns classes 0,...,9, hence +1.
    y_pred = np.argmax(probabilities, axis=1) + 1

    return np.mean(y_pred == y_true)

In [ ]:
# Wrapper function for final accuracy reporting
def test_classifier(model, loader, device):
    model.eval()

    all_probabilities = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = model(x)
            probabilities = torch.softmax(logits, dim=1)

            all_probabilities.append(probabilities.cpu().numpy())

            all_labels.append(y.numpy())

    probabilities = np.concatenate(all_probabilities, axis=0)
    y_true = np.concatenate(all_labels, axis=0)

    accuracy = accuracy_at_1(
        y_true,
        probabilities
    )

    return accuracy

In [ ]:
def test_classifier(model, loader, device):
    model.eval()

    all_probabilities = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = model(x)
            probabilities = torch.softmax(logits, dim=1)

            all_probabilities.append(probabilities.cpu().numpy())
            all_labels.append(y.numpy())

    probabilities = np.concatenate(all_probabilities, axis=0)

    # Shift label scale
    y_true = np.concatenate(all_labels, axis=0) + 1

    accuracy = accuracy_at_1(
        y_true,
        probabilities
    )

    return accuracy

In [ ]:
def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

In [ ]:
encoder_params = count_trainable_parameters(autoencoder.encoder)
reconstructor_params = count_trainable_parameters(autoencoder.reconstructor)
autoencoder_params = count_trainable_parameters(autoencoder)
classifier_head_params = count_trainable_parameters(classifier.classifier)
final_model_params = count_trainable_parameters(classifier)

print("----- Trainable Parameters -----")
print(f"Encoder:              {encoder_params:,}")
print(f"Reconstructor:        {reconstructor_params:,}")
print(f"Autoencoder:          {autoencoder_params:,}")
print(f"Classification Head:  {classifier_head_params:,}")
print(f"Final Model:          {final_model_params:,}")


test_acc = test_classifier(
    classifier,
    test_loader,
    DEVICE
)

print("----- Accuracy -----")
print(f"Test Accuracy@1:      {test_acc * 100:.2f}%")

## Conclusion